# Endurance Pharmacology Explorer — Chai-1 Inference

**Run this notebook on Google Colab with GPU runtime (T4 minimum, A100 recommended).**

This notebook runs Chai-1 structure predictions for 6 endurance-relevant drug-target pairs.

## Run Order (Progressive Validation)
1. **Ivabradine-HCN4** (control) — validates pipeline works. Check output before continuing.
2. **Cardarine-PPARδ, Roxadustat-PHD2** (standard small molecule docking)
3. **ITPP-Hemoglobin** (tetramer + allosteric ligand — needs more memory)
4. **BPC-157-VEGFR2, TB-500-Actin** (peptide-protein — hardest, run last)

In [ ]:
# Step 1: Install Chai-1
!pip install chai_lab==0.6.1 pyyaml -q

In [ ]:
# Step 2: Upload FASTA files
# Option A: Upload from local machine
from google.colab import files
import os

os.makedirs('fasta', exist_ok=True)
os.makedirs('results', exist_ok=True)

print('Upload all .fasta files from data/fasta/ directory:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'fasta/{name}', 'wb') as f:
        f.write(data)
    print(f'  Saved: fasta/{name}')

In [ ]:
# Step 3: Verify FASTA files
from pathlib import Path

fasta_dir = Path('fasta')
fasta_files = sorted(fasta_dir.glob('*.fasta'))
print(f'Found {len(fasta_files)} FASTA files:\n')

for f in fasta_files:
    content = f.read_text()
    n_chains = content.count('>protein|')
    n_ligands = content.count('>ligand|')
    print(f'  {f.name}: {n_chains} protein chain(s), {n_ligands} ligand(s)')

assert len(fasta_files) == 6, f'Expected 6 FASTA files, got {len(fasta_files)}'

In [ ]:
import torch
from pathlib import Path
from chai_lab.chai1 import run_inference

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Phase 1: Control — Ivabradine + HCN4
Run this first. If the output looks reasonable, proceed to Phase 2.

In [ ]:
# Run control: Ivabradine-HCN4
import shutil

output_dir = Path('results/ivabradine_hcn4')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/ivabradine_hcn4.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)

print(f'\nGenerated {len(list(output_dir.glob("*.cif")))} predictions')
print('Check output before continuing to Phase 2!')

## Phase 2: Frontier Small Molecules

In [ ]:
# Run Cardarine-PPARdelta
import shutil

output_dir = Path('results/cardarine_ppard')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/cardarine_ppard.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)
print(f'Cardarine-PPARd: {len(list(output_dir.glob("*.cif")))} predictions')

In [ ]:
# Run Roxadustat-PHD2
import shutil

output_dir = Path('results/roxadustat_phd2')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/roxadustat_phd2.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)
print(f'Roxadustat-PHD2: {len(list(output_dir.glob("*.cif")))} predictions')

In [ ]:
# Run ITPP-Hemoglobin (tetramer — may need A100 for memory)
import shutil

output_dir = Path('results/itpp_hemoglobin')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/itpp_hemoglobin.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)
print(f'ITPP-Hemoglobin: {len(list(output_dir.glob("*.cif")))} predictions')

## Phase 3: Frontier Peptides (Hardest)

In [ ]:
# Run BPC-157 + VEGFR2 (large protein — A100 recommended)
import shutil

output_dir = Path('results/bpc157_vegfr2')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/bpc157_vegfr2.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)
print(f'BPC-157-VEGFR2: {len(list(output_dir.glob("*.cif")))} predictions')

In [ ]:
# Run TB-500 + G-actin
import shutil

output_dir = Path('results/tb500_actin')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

candidates = run_inference(
    fasta_file=Path('fasta/tb500_actin.fasta'),
    output_dir=output_dir,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    seed=42,
    device=torch.device('cuda:0'),
    use_esm_embeddings=True,
    use_msa_server=True,
)
print(f'TB-500-Actin: {len(list(output_dir.glob("*.cif")))} predictions')

## Step 4: Download Results
Download the entire results directory to your local machine.

In [ ]:
import shutil

# List all outputs
results_root = Path('results')
for pair_dir in sorted(results_root.iterdir()):
    if pair_dir.is_dir():
        outputs = list(pair_dir.glob('*'))
        print(f'{pair_dir.name}: {len(outputs)} files')

# Zip for download
shutil.make_archive('chai1_results', 'zip', 'results')
print('\nDownload chai1_results.zip and extract into endurance-pharma-explorer/results/')
files.download('chai1_results.zip')